# 05 · Feature lists for the two models

One shared candidate list for both models, then some extra trimming for Elastic net only.

- A. Shared candidates: drop columns no model should use (leakage, almost one value). No models are fit
  here.
- B. Elastic net: keep one form of each duplicate and remove high VIF, so the coefficients stay readable. Its
  L1 penalty takes care of weak columns while it fits.
- C. Random forest: uses the shared list as is. Trees don't mind correlated or unused columns (trimming its
  list in an earlier version of this notebook changed MAE by at most 0.001). Permutation importance is only used to explain it.

Both models get the same information, so if they disagree on a listing, it's because of the model, not the
inputs.

Setup: 2026-09 snapshots, with Kandice's review sentiment features and Kent's distance to the nearest tourist
attraction (same data as 04_baseline), GroupKFold(5)
by host, MAE on log price.

## A0. Setup

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="Pandas requires version")  # Anaconda 'bottleneck' noise
warnings.filterwarnings("ignore", message=".*not a team snapshot.*")
sys.path.insert(0, "..")   # ".." is the folder above notebooks/, i.e. the repo root, where strfair/ lives
import strfair             # the team's shared data pipeline

USE_SENTIMENT = True        # strfair.prepare() with Kandice's sentiment features (slow the first time per city)

# the September 2026 snapshot of each city, same as 04_baseline
SNAPSHOTS = {"new-york-city": "2026-09-14",
             "los-angeles": "2026-09-14",
             "toronto": "2026-09-16",
             "san-diego": "2026-09-20",
             "clark-county-nv": "2026-09-20",
             "jersey-city": "2026-09-21"}

OUT = strfair.REPO_ROOT / "docs" / "eda" / "feature_lists"   # every table is saved here
OUT.mkdir(parents=True, exist_ok=True)

# same as 04_baseline. sentiment_review_count is a review count under another name, so it is never a candidate.
# nearest_attraction is only the attraction's name, kept for checking; the distance columns carry the information
drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw", "discount_amount",
                                                                         "sentiment_review_count", "nearest_attraction"]

# quote control variables: they say *when* and *for how long* the quote was taken, so they stay in both models
QUOTE_CONTROLS = ["is_monthly_quote", "quote_nights", "lead_band", "min_nights_band", "weekend_checkin",
                  "checkin_dow", "checkin_month"]

In [2]:
from strfair.data import load_listings
from strfair.prep import _sample_mask


# X, y and host groups for one city, with the same columns dropped as in 04_baseline
def load_city(city, snapshot):
    if USE_SENTIMENT:
        data = strfair.prepare(city, snapshot)          # read from the cache after the first run
        sample, F = data.sample, data.features
    else:
        # same rows and features as strfair.prepare(), without the sentiment and tourist steps and without the cache
        listings = load_listings(city, snapshot)
        sample = listings[_sample_mask(listings)].reset_index(drop=True)
        F = strfair.build_features(sample, city).reset_index(drop=True)

    X = F[[c for c in F.columns if c not in drop]].copy()
    if "avg_sentiment" in X:
        X["no_sentiment"] = X["avg_sentiment"].isna().astype(int)   # 1 = fewer than 3 usable reviews
    y = F["log_price"]                                              # target: log of the nightly price
    groups = sample["host_id"].values                               # one host's listings stay in the same fold
    return sample, X, y, groups


cities = {}     # city -> (X, y, groups)
family = {}     # column -> group: size, location, host, reviews, ...

for city, snapshot in SNAPSHOTS.items():
    sample, X, y, groups = load_city(city, snapshot)
    cities[city] = (X, y, groups)

    # which feature family each column comes from. sentiment and tourist columns are added in prep.py,
    # not features.py, so they get their group here
    fam = strfair.feature_families(sample)
    for col in X.columns:
        if col in fam.index:
            family[col] = fam[col]
        elif col in ("dist_to_nearest_attraction_km", "dist_exp_decay"):
            family[col] = "tourist"
        else:
            family[col] = "sentiment"

    # nothing from LEAKY or DEMAND_RESPONSE may be left
    assert not set(X.columns) & set(strfair.LEAKY + strfair.DEMAND_RESPONSE), city

family = pd.Series(family)
print(len(family), "columns before screening")
family.value_counts()

new-york-city 2026-09-14 | listings 29,918 rows, 19,396 in sample | features 119 cols | code 8fd202ea | data 8d904721  (cached)


los-angeles 2026-09-14 | listings 43,781 rows, 36,663 in sample | features 119 cols | code 8fd202ea | data c42b65f1  (cached)


toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 118 cols | code 8fd202ea | data 93c4b714  (cached)


san-diego 2026-09-20 | listings 13,253 rows, 11,614 in sample | features 118 cols | code 8fd202ea | data 23c71b6d  (cached)


clark-county-nv 2026-09-20 | listings 20,651 rows, 17,440 in sample | features 118 cols | code 8fd202ea | data 2a715a57  (cached)


jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 118 cols | code 8fd202ea | data 460245b9  (cached)


91 columns before screening


amenities       18
reviews         14
host            12
listing_text    12
target_quote    11
size            10
location         6
sentiment        5
tourist          2
availability     1
Name: count, dtype: int64

## A1. Data quality and leakage check

The first cell stops if any column has Spearman |ρ| of 0.9 or more with log price, which usually means it was
computed from the price. (discount_amount was dropped for this reason; discount_share and has_discount were
checked in an earlier version of this notebook and are fine.)

Columns where 95% or more of listings share one value in at least 4 cities are dropped. Missing values are
only shown here; both models fill them with the median.

In [3]:
# leakage check: stops if any column tracks log price almost perfectly
for city, (X, y, groups) in cities.items():
    rho = X.select_dtypes("number").corrwith(y, method="spearman").abs()   # rank correlation, sign ignored
    rho = rho.dropna()                       # columns with one value in this city have no correlation
    assert (rho < 0.9).all(), f"{city}: check {list(rho[rho >= 0.9].index)} for leakage"

/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/anaconda3/envs/airbnb-project/lib/python3.11/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [4]:
# share of missing values per column and city
missing = {}
for city, (X, y, groups) in cities.items():
    missing[city] = X.isna().mean()
missing = pd.DataFrame(missing)

print("columns with more than 20% missing in some city:")
missing[missing.max(axis=1) > 0.2].round(2)

columns with more than 20% missing in some city:


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city
avg_sentiment,0.42,0.38,0.33,0.24,0.42,0.31
months_since_last_review,0.26,0.24,0.18,0.15,0.27,0.16
negative_review_pct,0.42,0.38,0.33,0.24,0.42,0.31
review_scores_accuracy,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_checkin,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_cleanliness,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_communication,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_location,0.26,0.24,0.18,0.15,0.27,0.16
review_scores_rating,0.26,0.24,0.18,0.15,0.27,0.16
sentiment_std,0.42,0.38,0.33,0.24,0.42,0.31


In [5]:
# share of the most common value in a column, e.g. 0.98 if 98% of listings have Wi-Fi
def most_common_share(column):
    shares = column.value_counts(normalize=True, dropna=False)   # share of each value, largest first
    return shares.iloc[0]


top_share = {}
for city, (X, y, groups) in cities.items():
    shares = {}
    for col in X.columns:
        shares[col] = most_common_share(X[col])
    top_share[city] = pd.Series(shares)
top_share = pd.DataFrame(top_share)     # one row per column, one column per city

# almost constant: one value covers 95%+ of listings in at least 4 cities
n_cities = (top_share >= 0.95).sum(axis=1)
almost_constant = n_cities[n_cities >= 4].index.tolist()
print("almost one value in 4+ cities:", almost_constant)
top_share.loc[almost_constant].round(3)

almost one value in 4+ cities: ['am_breakfast', 'am_smoking_allowed', 'am_wifi', 'amg_breakfast', 'amg_connectivity', 'has_description', 'has_neighborhood_overview', 'host_has_profile_pic', 'host_identity_verified']


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city
am_breakfast,0.955,0.965,0.985,0.976,0.965,0.975
am_smoking_allowed,0.979,0.968,0.988,0.983,0.961,0.994
am_wifi,0.983,0.988,0.991,0.991,0.986,0.998
amg_breakfast,0.997,0.999,1.000,0.997,1.000,1.000
amg_connectivity,0.984,0.988,0.991,0.992,0.986,0.999
has_description,0.996,0.996,0.997,0.999,0.998,0.999
has_neighborhood_overview,1.000,1.000,1.000,1.000,1.000,1.000
host_has_profile_pic,0.956,0.967,0.958,0.980,0.959,0.967
host_identity_verified,0.984,0.987,0.983,0.973,0.983,0.980


Result: the leakage check passes everywhere. 9 columns are dropped for having almost one value: am_wifi,
amg_connectivity, am_breakfast, amg_breakfast, am_smoking_allowed, host_identity_verified,
host_has_profile_pic, has_description, has_neighborhood_overview. Missing values: the review scores and
months_since_last_review (15-27%, the listings with no reviews) and the sentiment columns (24-42%, the listings
with fewer than 3 usable reviews; no_sentiment flags them).

## A2. Column types

checkin_dow and checkin_month are categories (Saturday isn't 5 times Monday), so they're converted to text.

In [6]:
# day of week and month are labels, not amounts: as text they get one-hot columns in Elastic net
for city, (X, y, groups) in cities.items():
    X["checkin_dow"] = X["checkin_dow"].astype(str)
    X["checkin_month"] = X["checkin_month"].astype(str)

# every text column the models will encode
categorical = sorted(set(strfair.CATEGORICAL + ["checkin_dow", "checkin_month"]) & set(family.index))
print("categorical columns:", categorical)

categorical columns:

 ['checkin_dow', 'checkin_month', 'lead_band', 'min_nights_band', 'neighbourhood', 'neighbourhood_group', 'property_group', 'room', 'room_type']


## A3. Shared candidate list

What's left after A1. Both models start from here.

In [7]:
# the shared list: everything except the almost-constant columns from A1
shared_drop = set(almost_constant)
candidates = [c for c in family.index if c not in shared_drop]

# save the list with each column's group, then count columns per group
pd.DataFrame({"column": candidates, "group": family[candidates].values}).to_csv(OUT / "candidates.csv", index=False)
print(f"{len(family)} columns -> {len(candidates)} candidates")
family[candidates].value_counts()

91 columns -> 82 candidates


reviews         14
amenities       13
target_quote    11
size            10
host            10
listing_text    10
location         6
sentiment        5
tourist          2
availability     1
Name: count, dtype: int64

## B. Elastic net

Only Elastic net needs this, because its coefficients are what we report. If two columns carry the same
information, the weight gets split between them and the coefficients stop making sense. The forest doesn't
have that problem.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


# same Elastic net as 04_baseline. Wrapped in a pipeline, so the median, scaling and one-hot levels are
# learned from the training folds only
def make_enet(X):
    cat = list(X.select_dtypes(exclude="number").columns)   # text columns, e.g. room_type, neighbourhood
    num = [c for c in X.columns if c not in cat]

    # numbers: fill missing values with the median, then scale
    # categories: one column per level, levels with < 20 listings are grouped together
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    # alpha: overall penalty strength; l1_ratio=0.5: half L1 (can set weights to 0), half L2. Not tuned yet
    return make_pipeline(pre, ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))


# same forest as 04_baseline, but 100 trees instead of 300 to keep the run short
def make_rf(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # trees don't need scaling or one-hot columns: each category just gets a number
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), cat),
    ])
    # 2 cores per forest: the 5 folds run in parallel, 5 x 2 = 10 cores
    return make_pipeline(pre, RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                                    n_jobs=2, random_state=42))


gkf = GroupKFold(n_splits=5)   # the folds only depend on the hosts, so every run uses the same folds


# a new model fit on each fold, MAE on the held-out fold; the 5 folds run in parallel
def run_model(make_model, X, y, groups):
    scores = cross_val_score(make_model(X), X, y, groups=groups, cv=gkf,
                             scoring="neg_mean_absolute_error", n_jobs=5)
    return -scores             # MAE of each fold

### B1. One form of each column

Keep one version of each column. A few exceptions, all because Elastic net did worse without them in NYC, LA
or San Diego: room_type, lead_days (lead_band stays too, as a quote control), the coordinates (a linear model
can still pick up "closer to the coast, more expensive"), and host_listings_count. That last one isn't a
duplicate: it counts the host's listings on all of Airbnb, while log_host_listings only counts this city.

| columns | keep |
|---|---|
| km_to_centre, log_km_to_centre | log_km_to_centre |
| number_of_reviews, log_reviews | log_reviews |
| calculated_host_listings_count, log_host_listings, multi_host, single_listing_host | log_host_listings |
| review_scores_rating, rating_filled | rating_filled |
| no_reviews, no_rating (the same listings) | no_rating |
| accommodates, guests_per_bedroom, guests_per_bed | accommodates |
| room_type, room | room_type |
| minimum_nights, min_nights_band | min_nights_band |
| dist_to_nearest_attraction_km, dist_exp_decay | dist_exp_decay |

In [9]:
# the other versions of a column that Elastic net drops (the table above says which one stays)
other_forms = ["km_to_centre", "number_of_reviews", "calculated_host_listings_count",
               "multi_host", "single_listing_host", "guests_per_bedroom", "guests_per_bed",
               "review_scores_rating", "no_reviews", "room", "minimum_nights",
               "dist_to_nearest_attraction_km"]

enet_cols = [c for c in candidates if c not in other_forms]
print(f"{len(candidates)} candidates -> {len(enet_cols)} after B1")

82 candidates -> 70 after B1


### B2. VIF

Drop the column with the highest VIF (median over the 6 cities), recompute, and repeat until nothing is above
10. Quote controls are never dropped. Numeric columns only.

In [10]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


# VIF of every column in X, computed in one city
def vif(X):
    num = X.fillna(X.median())               # VIF can't handle missing values
    num = num.loc[:, num.std() > 0]          # skip columns with only one value in this city
    num = add_constant(num)                  # VIF needs an intercept column ("const")

    values = {}
    for i, col in enumerate(num.columns):
        if col != "const":
            values[col] = variance_inflation_factor(num.values, i)
    return pd.Series(values)


# VIF only works on numbers, so the text columns are left out
numeric = [c for c in enet_cols if c not in categorical]
vif_removed = []

# drop the worst column, recompute, repeat until every median VIF is 10 or less
while True:
    table = {}
    for city, (X, y, groups) in cities.items():
        table[city] = vif(X[[c for c in numeric if c in X.columns]])
    median_vif = pd.DataFrame(table).median(axis=1)       # median over the 6 cities

    # quote controls are never removed
    can_remove = median_vif.drop(QUOTE_CONTROLS, errors="ignore")
    worst = can_remove.idxmax()
    if can_remove[worst] <= 10:
        break
    print(f"remove {worst} (median VIF {can_remove[worst]:.1f})")
    vif_removed.append(worst)
    numeric.remove(worst)

enet_cols = [c for c in enet_cols if c not in vif_removed]
print(f"\n{len(enet_cols)} columns after B2. Highest median VIF left:")
median_vif.sort_values(ascending=False).head(10).round(1)

remove avg_sentiment (median VIF 20.7)



69 columns after B2. Highest median VIF left:


log_reviews           7.5
rating_filled         6.8
log_km_to_centre      6.6
am_free_parking       6.2
dist_exp_decay        5.7
amg_parking           5.6
accommodates          5.4
sentiment_std         5.3
bedrooms              5.1
weighted_sentiment    5.0
dtype: float64

Result: one column is dropped, avg_sentiment (median VIF 20.7): it is almost the same number as
weighted_sentiment, which stays. Everything left is under 10; the highest is log_reviews at 7.5. accommodates and
log_host_listings were at 19 and 47 before B1 removed the columns derived from them; now they're 5.4 and under 5 (the 19 and 47 are from an earlier run).

### B3. Check: is the trimmed list as accurate as all candidates?

Elastic net with all candidates vs the trimmed list, on the same folds. The trimmed list passes if it isn't
worse by more than the std of the 5 fold differences.

In [11]:
# Elastic net with all candidates vs the trimmed list, on the same 5 folds
rows = []
for city, (X, y, groups) in cities.items():
    cols = [c for c in enet_cols if c in X.columns]
    full = run_model(make_enet, X[[c for c in candidates if c in X.columns]], y, groups)
    trimmed = run_model(make_enet, X[cols], y, groups)
    diff = trimmed - full                       # per fold; positive = the trimmed list is worse
    # ok: the average change is no bigger than the fold-to-fold noise
    rows.append([city, len(cols), full.mean(), trimmed.mean(), diff.mean(), diff.mean() <= diff.std()])

enet_check = pd.DataFrame(rows, columns=["city", "columns", "all_candidates", "trimmed", "change", "ok"])
enet_check.to_csv(OUT / "enet_check.csv", index=False)
enet_check.round(4)

,city,columns,all_candidates,trimmed,change,ok
0,new-york-city,69,0.2923,0.2950,0.0028,False
1,los-angeles,69,0.2804,0.2823,0.0018,False
2,toronto,68,0.2492,0.2497,0.0005,True
3,san-diego,68,0.2817,0.2832,0.0014,True
4,clark-county-nv,68,0.3462,0.3482,0.0020,True
5,jersey-city,68,0.2236,0.2241,0.0005,True


Result: 69 columns (68 in cities without neighbourhood_group). The `change` column above is the MAE
difference vs all candidates: +0.0005 to +0.0028, under 1% of the MAE in every city. NYC and LA miss the bar
only because their folds are very close to each other, so even a tiny change counts. Keeping the raw
minimum_nights didn't help: it moves with quote_nights (Spearman 0.97) and VIF drops it (from an earlier run).

## C. Random forest

Uses all shared candidates, including the coordinates and the other forms of each column. It has 100 trees
instead of the 300 in 04_baseline to keep the run short, so its MAE differs a little from 04.

In [12]:
rf_cols = list(candidates)     # the forest keeps every shared candidate

rows = []
for city, (X, y, groups) in cities.items():
    cols = [c for c in rf_cols if c in X.columns]
    rf_mae = run_model(make_rf, X[cols], y, groups)   # MAE of each of the 5 folds
    # Elastic net MAE from B3, to see how much accuracy the interpretable model gives up
    enet_mae = enet_check.loc[enet_check["city"] == city, "trimmed"].iloc[0]
    rows.append([city, len(cols), rf_mae.mean(), enet_mae, enet_mae - rf_mae.mean()])

rf_check = pd.DataFrame(rows, columns=["city", "columns", "rf_mae", "enet_trimmed_mae", "accuracy_given_up"])
rf_check.to_csv(OUT / "rf_check.csv", index=False)
rf_check.round(4)

,city,columns,rf_mae,enet_trimmed_mae,accuracy_given_up
0,new-york-city,82,0.2809,0.2950,0.0141
1,los-angeles,82,0.2686,0.2823,0.0137
2,toronto,81,0.2459,0.2497,0.0038
3,san-diego,81,0.2759,0.2832,0.0072
4,clark-county-nv,81,0.3086,0.3482,0.0396
5,jersey-city,81,0.2300,0.2241,-0.0059


Result: 82 columns (81 in four cities). MAE the interpretable model gives up vs the forest: 0.014 in NYC, 0.014
in LA, 0.004 in Toronto, 0.007 in San Diego and 0.040 in Las Vegas. In Jersey City Elastic net is 0.006 better.
Las Vegas has the biggest gap, probably because its location effect isn't linear and the forest can split on
the coordinates.

### C1. Permutation importance

Train on 4 folds, shuffle one column at a time in the held-out fold, and see how much MAE goes up. One fold per
city is enough for a ranking. This is only for explaining the forest, not for dropping columns. The forest's
built-in feature_importances_ isn't used: it's computed on the training data and favours columns with many
values.

In [13]:
# which columns the forest relies on most, per city
from sklearn.inspection import permutation_importance

importance = {}
for city, (X, y, groups) in cities.items():
    Xr = X[[c for c in rf_cols if c in X.columns]]
    train_idx, test_idx = next(gkf.split(Xr, y, groups))    # the first fold only: enough for a ranking
    model = make_rf(Xr).fit(Xr.iloc[train_idx], y.iloc[train_idx])
    # shuffle one column at a time in the held-out fold; importance = how much MAE goes up (3 shuffles each)
    result = permutation_importance(model, Xr.iloc[test_idx], y.iloc[test_idx],
                                    scoring="neg_mean_absolute_error", n_repeats=3, random_state=42)
    importance[city] = pd.Series(result.importances_mean, index=Xr.columns)
    print(city, "done")

importance = pd.DataFrame(importance)
# median over the cities, so one city can't dominate the ranking
importance["median"] = importance.median(axis=1)
importance = importance.sort_values("median", ascending=False)
importance.round(4).to_csv(OUT / "rf_permutation_importance.csv")
importance.head(20).round(4)

new-york-city done


los-angeles done


toronto done


san-diego done


clark-county-nv done


jersey-city done


,new-york-city,los-angeles,toronto,san-diego,clark-county-nv,jersey-city,median
quote_nights,0.1889,0.1293,0.2544,0.0419,0.1461,0.0894,0.1377
room_type,0.0699,0.0476,0.0937,0.0013,0.0013,0.1136,0.0588
baths,0.0257,0.0937,0.0447,0.0333,0.0429,0.0315,0.0381
accommodates,0.0372,0.0284,0.0315,0.1875,0.1893,0.0309,0.0344
bedrooms,0.0418,0.1425,0.0189,0.0237,0.0052,0.0232,0.0234
longitude,0.0220,0.0585,0.0015,0.0310,0.0093,0.0109,0.0165
shared_bath,0.0212,0.0078,0.0089,0.0077,0.0081,0.0011,0.0079
host_listings_count,0.0116,0.0040,0.0006,0.0023,0.0142,0.0130,0.0078
lead_days,0.0717,0.0049,0.0475,0.0029,0.0033,0.0104,0.0077
minimum_nights,0.0034,0.0291,0.0001,0.0106,0.0009,0.0413,0.0070


Result: quote_nights is first overall (median 0.138; 0.25 in Toronto, 0.19 in NYC), except in San Diego and
Las Vegas, where guests (accommodates) comes first. Prices are per night for
one quote and monthly quotes are discounted, so quote length matters most, which is why the quote controls stay
in both models. Size comes next: room type (0.11 in Jersey City, 0.09 in Toronto, 0.07 in NYC), bathrooms, guests
(0.19 in San Diego and Las Vegas) and bedrooms (0.14 in LA). Then location: longitude (0.06 in LA) and, in NYC,
distance to the nearest attraction (0.06 for dist_exp_decay, 0.03 for the km). Host, review, amenity and text
columns are all 0.015 or less, and the sentiment columns about 0.002 or less. Correlated columns share their
importance: in the run without attractions, km_to_centre and log_km_to_centre each got about 0.04 in NYC; now
they get 0.002, because the attraction distance tells the forest the same thing (Midtown is both).

## Final lists

In [14]:
# why a column is missing from a list
def reason(col):
    if col in almost_constant:
        return "almost one value (A1)"
    if col in other_forms:
        return "Elastic net: other form of a column (B1)"
    if col in vif_removed:
        return "Elastic net: VIF > 10 (B2)"
    return ""


# one row per column: which lists it is in, and why not
rows = []
for col in family.index:
    rows.append([col, family[col], col in candidates, col in enet_cols, col in rf_cols, reason(col)])
decisions = pd.DataFrame(rows, columns=["column", "group", "candidate", "elastic_net", "random_forest", "reason"])
decisions.to_csv(OUT / "decisions.csv", index=False)

# the lists that go into strfair/models.py
print("CANDIDATES =", candidates)
print("ENET_FEATURES =", enet_cols)
print("RF_FEATURES =", rf_cols)
decisions[decisions["reason"] != ""]

CANDIDATES = ['lead_days', 'quote_nights', 'weekend_checkin', 'checkin_dow', 'checkin_month', 'is_monthly_quote', 'lead_band', 'has_discount', 'discount_share', 'minimum_nights', 'min_nights_band', 'room_type', 'room', 'property_group', 'accommodates', 'bedrooms', 'beds', 'shared_bath', 'baths', 'guests_per_bedroom', 'guests_per_bed', 'latitude', 'longitude', 'km_to_centre', 'log_km_to_centre', 'neighbourhood', 'neighbourhood_group', 'superhost', 'host_listings_count', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'log_host_listings', 'multi_host', 'single_listing_host', 'host_entire_home_share', 'host_years', 'number_of_reviews', 'log_reviews', 'no_reviews', 'tenure_months', 'new_listing', 'months_since_last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'no_rating', 'rating_filled'

,column,group,candidate,elastic_net,random_forest,reason
9,minimum_nights,target_quote,True,False,True,Elastic net: other form of a column (B1)
12,room,size,True,False,True,Elastic net: other form of a column (B1)
19,guests_per_bedroom,size,True,False,True,Elastic net: other form of a column (B1)
20,guests_per_bed,size,True,False,True,Elastic net: other form of a column (B1)
23,km_to_centre,location,True,False,True,Elastic net: other form of a column (B1)
28,host_identity_verified,host,False,False,False,almost one value (A1)
29,host_has_profile_pic,host,False,False,False,almost one value (A1)
31,calculated_host_listings_count,host,True,False,True,Elastic net: other form of a column (B1)
35,multi_host,host,True,False,True,Elastic net: other form of a column (B1)
36,single_listing_host,host,True,False,True,Elastic net: other form of a column (B1)


## Findings

2026-09 snapshots, with Kandice's sentiment features and Kent's attraction distance.

- **Shared candidates:** 82 of 91 columns. Dropped: nine columns with almost one value (Wi-Fi, breakfast,
  smoking, verified ID, profile picture, description flags).
- **Elastic net (69 columns):** one form of each column, plus `room_type`, `lead_days`, the coordinates and
  `host_listings_count`, which it needs in NYC, LA and San Diego. VIF drops `avg_sentiment` (a copy of
  `weighted_sentiment`); nothing else is above 10. Of the two attraction columns it keeps `dist_exp_decay`.
  Within 0.003 MAE of all candidates in every city.
- **Random forest (82 columns):** all shared candidates. It is 0.004–0.040 MAE better than Elastic net in 5
  cities (most in Las Vegas) and slightly worse in Jersey City.
- **What drives price for the forest:** size and quote length (median 42% and 40% of the importance
  per city), then location (7%). Host, reviews, amenities and listing text add little on their own.
- **Attraction distance:** 14% of the forest's importance in NYC, about 1% or less elsewhere. In
  04_baseline it changes MAE by at most 0.003.
- **Sentiment:** the sentiment columns get 0.4% of the forest's importance, and in 04_baseline they change
  MAE by at most 0.003. They are kept in the lists, but they barely matter for price.